# NFL Play Transformer -- Colab setup

Before running any cells: **Runtime -> Change runtime type -> T4 GPU**, then Save.

T4 is enough here -- the model is ~1.2M parameters, nowhere near needing an A100/L4/H100.

**One-time setup (do this once, outside this notebook):**
1. Create a GitHub Personal Access Token with `repo` scope: https://github.com/settings/tokens -> Generate new token (classic) -> check `repo`.
2. In this Colab session, click the key icon (Secrets) in the left sidebar.
3. Add a new secret named `GITHUB_TOKEN` with that token as the value, and toggle notebook access on.

This notebook never sees or stores the token in its own text -- it's pulled from Colab's Secrets manager at runtime.

## 1. Confirm the GPU is attached

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv


## 2. Clone the private repo (using the GITHUB_TOKEN secret)

In [ ]:
from google.colab import userdata
token = userdata.get('GITHUB_TOKEN')

!git clone https://{token}@github.com/havishs/nfl-play-transformer.git
%cd nfl-play-transformer


## 3. Install dependencies

torch/pandas/numpy/pyarrow are already present in Colab (with CUDA-enabled torch).
nfl_data_py pins pandas<2.0/numpy<1.0, which is stale -- installed with `--no-deps`
so it doesn't try to downgrade Colab's own pandas/numpy.

Not pinning `requests` here: Colab's own `google.colab` package (used by
`drive.mount()` below) depends on a specific requests version, and pinning
to the version resolved locally overwrites it and breaks Drive mounting.
Colab's existing requests is new enough for nfl_data_py's plain HTTP fetches.

In [ ]:
!pip install -q nfl_data_py==0.3.3 --no-deps
!pip install -q appdirs==1.4.4

import torch
print('cuda available:', torch.cuda.is_available())
print('device name:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none')

## 4. Mount Google Drive

Colab's local disk is wiped when the runtime disconnects/recycles. The dataset
cache (~1GB) and checkpoint are worth keeping across sessions, so this mounts
Drive and we'll copy results there after training.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_DIR = '/content/drive/MyDrive/nfl-play-transformer'
os.makedirs(DRIVE_DIR, exist_ok=True)


## 5. Fetch data

`data/` is gitignored (not in the repo), so this re-fetches the 2018-2023
seasons via nfl_data_py -- same script already validated locally. If you'd
rather not re-download, you can instead upload the local `data/` folder to
`DRIVE_DIR/data` once and symlink it here -- ask if you want that path instead.

In [ ]:
%cd pipeline
!python fetch_data.py 2018 2019 2020 2021 2022 2023


## 6. Restore a cached dataset build from Drive, if one exists

Skips the ~18-20 min `iterrows()` rebuild on repeat runs. First run this
session: nothing to restore yet, this cell just no-ops.

In [ ]:
import glob, shutil
for f in glob.glob(f'{DRIVE_DIR}/dataset_cache_*.pkl'):
    shutil.copy(f, '.')
    print('restored', f)


## 7. Restore a cached checkpoint from Drive, if one exists

Skips training entirely if a checkpoint from a previous session is available.
First run this session: nothing to restore yet, this cell just no-ops. If you
want to evaluate an already-trained model (step 10 below) without retraining,
run this cell and then skip step 8 (Train).

In [ ]:
import glob, shutil
for f in glob.glob(f'{DRIVE_DIR}/checkpoint.pt'):
    shutil.copy(f, '.')
    print('restored', f)


## 8. Train

**Optional -- skip this cell if you just restored a checkpoint in step 7 and
only want to run the evaluation in step 10.** Running this trains a fresh
model from scratch and overwrites checkpoint.pt.

Device selection in train.py already checks `cuda` after `mps`, so this picks
up the T4 automatically -- no code changes needed for the GPU switch itself.

In [ ]:
!python train.py


## 9. Persist training results to Drive

Only needed if you trained in step 8 above -- skip if you restored an existing
checkpoint in step 7 and didn't retrain.

In [ ]:
!cp checkpoint.pt {DRIVE_DIR}/
!cp dataset_cache_*.pkl {DRIVE_DIR}/
print('saved to', DRIVE_DIR)


## 10. Evaluate win probability

Runs the Monte Carlo win-probability harness (`pipeline/win_prob_eval.py`)
against whatever `checkpoint.pt` is present in this directory -- either the
one you just trained in step 8, or the one restored from Drive in step 7.
Prints a per-quarter accuracy/Brier table and writes `win_prob_results.csv`.

Samples `N_EVAL_GAMES=25` validation games by default (a deliberately
conservative subsample -- see the module docstring for why). Adjust that
constant in `win_prob_eval.py` for a fuller run once you've seen how long
this takes.

In [ ]:
!python win_prob_eval.py


## 11. Persist evaluation results to Drive

In [ ]:
!cp win_prob_results.csv {DRIVE_DIR}/
print('saved to', DRIVE_DIR)
